# Projection bake-off refresh: 2026 with fresh data

`projection_bakeoff.ipynb` runs on the saved fixture in `reference/fixture/`, captured on Friday October 16, 2026, during week 6, so it can
hold out only weeks 4 and 5. This notebook re-runs **the same comparison on data fetched fresh from Sleeper**, so every week that has
finished since (week 6 after Monday night, October 19, and later weeks as the season goes on) becomes a test week. The fixture is only
read, never written: fresh responses go to `notebooks/.cache/sleeper/` (git-ignored).

Two extra checks use the fixture as a time capsule:

* **Did Sleeper change weeks 4-5 again after the capture?** Fresh weeks 4-5 against the fixture's 4.39.
* **How much does the after-the-fact rewrite flatter the MAE?** The fixture's week-6 projections were captured before every week-6 game
  except Thursday night's, so they are close to what the app showed before kickoff. Scoring them and the fresh (rewritten) week-6
  projections against the same final week-6 actuals measures the flattery directly.

Candidates, features, hyperparameters and test-week rule are identical to `projection_bakeoff.ipynb`.

## Data

`get()` is the cached fetch from `odds_calibration.ipynb`, with one change: a response is cached **only when its week is final** (every game
`complete` on Sleeper's scores endpoint). An in-progress week is fetched live and not stored, so a run on a Sunday afternoon cannot freeze
half a week of stats into the cache. The scores endpoint itself, the NFL state and the league are always fetched live.

As in the fixture notebook, projection rows start at week 2 (the fixture had no week-1 projections), so weeks 4 and 5 are evaluated with the
same training weeks as before. Week 1 stats still feed the form features.

In [1]:
import gzip, hashlib, json, re, time, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option('display.width', 140)
CACHE = Path('.cache') / 'sleeper'
CACHE.mkdir(parents=True, exist_ok=True)
BASE = 'https://api.sleeper.app'
FIX = Path('..') / 'reference' / 'fixture'          # read only

def fetch(path):
    for attempt in range(5):
        try:
            with urllib.request.urlopen(BASE + path, timeout=30) as r:
                return json.load(r)
        except urllib.error.HTTPError as e:
            if e.code == 404: return None
            time.sleep(1 + 2 * attempt)
        except Exception:
            time.sleep(1 + 2 * attempt)
    raise RuntimeError('failed: ' + path)

def get(path, cache=True):
    # GET a Sleeper path; cached forever on disk when cache=True (only ever passed for final weeks).
    f = CACHE / (hashlib.sha1(path.encode()).hexdigest()[:20] + '.json.gz')
    if cache and f.exists():
        with gzip.open(f, 'rt') as fh: return json.load(fh)
    data = fetch(path)
    if cache:
        with gzip.open(f, 'wt') as fh: json.dump(data, fh)
    return data

def get_many(paths, cache=True, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda p: get(p, cache), paths))

manifest = json.load(open(FIX / 'manifest.json'))
LEAGUE_ID, SEASON = manifest['league_id'], manifest['season']
CAPTURED = datetime.fromtimestamp(manifest['captured_at'] / 1000, timezone.utc)
FETCHED = datetime.now(timezone.utc)
state = get('/v1/state/nfl', cache=False)
league = get(f'/v1/league/{LEAGUE_ID}', cache=False)
CUR = int(state['week']) if state['season'] == SEASON else 18
WEEKS = list(range(1, CUR + 1))
scores = {w: s or [] for w, s in zip(WEEKS, get_many([f'/scores/nfl/regular/{SEASON}/{w}' for w in WEEKS], cache=False))}
final = {w: bool(scores[w]) and all(g.get('status') == 'complete' for g in scores[w]) for w in WEEKS}

POS = ['QB', 'RB', 'WR', 'TE', 'K', 'DEF']
POSI = {p: i for i, p in enumerate(POS)}
q = '&'.join(f'position[]={p}' for p in POS)
stats = {w: get(f'/v1/stats/nfl/regular/{SEASON}/{w}', cache=final[w]) or {} for w in WEEKS}
proj = {w: {r['player_id']: r for r in (get(f'/projections/nfl/{SEASON}/{w}?season_type=regular&{q}', cache=final[w]) or [])} for w in WEEKS}
print(league['name'], '| season', SEASON, '| NFL week', CUR, '| fixture captured', CAPTURED.strftime('%Y-%m-%d %H:%M UTC'),
      '| fresh data fetched', FETCHED.strftime('%Y-%m-%d %H:%M UTC'))
pd.DataFrame({w: dict(games=len(scores[w]), complete=sum(g.get('status') == 'complete' for g in scores[w]), final=final[w],
                      stat_lines=len(stats[w]), projection_rows=len(proj[w])) for w in WEEKS})

fantasy | season 2026 | NFL week 6 | fixture captured 2026-10-16 21:29 UTC | fresh data fetched 2026-10-18 23:02 UTC


,1,2,3,4,5,6
games,16,16,16,16,15,14
complete,16,16,16,16,15,10
final,True,True,True,True,True,False
stat_lines,2331,2352,2357,2386,2241,1786
projection_rows,3306,3306,3306,3306,3304,3302


## Scoring with the league's settings

Identical to `projection_bakeoff.ipynb`.

In [2]:
def make_scorer(sc):
    rec = sc.get('rec', 1)
    base = 'pts_ppr' if rec >= 0.75 else 'pts_half_ppr' if rec >= 0.25 else 'pts_std'
    pass_td = sc.get('pass_td', 4) - 4
    bonus = [(k, sc[k]) for k in ('bonus_rec_te', 'bonus_rec_wr', 'bonus_rec_rb') if sc.get(k)]
    def score(st):
        if not st: return 0.0
        v = st.get(base) if st.get(base) is not None else st.get('pts_ppr', 0)
        if pass_td and st.get('pass_td'): v += pass_td * st['pass_td']
        for k, w in bonus:
            if st.get(k): v += w * st[k]
        return v
    return score
SC = league['scoring_settings']
score = make_scorer(SC)
print('rec =', SC.get('rec', 1), '| pass_td =', SC.get('pass_td', 4), '| catch bonuses:', {k: SC[k] for k in SC if k.startswith('bonus_rec') and SC[k]})

rec = 1.0 | pass_td = 4.0 | catch bonuses: {}


## Player-week table

Same loading rules and features as `projection_bakeoff.ipynb`; only final weeks produce rows.

In [3]:
info = {}
for w in sorted(proj, reverse=True):
    for pid, r in proj[w].items():
        if pid in info: continue
        pl = r.get('player') or {}
        info[pid] = dict(pos=pl.get('position') or ('DEF' if pid.isalpha() else None), team=pl.get('team') or r.get('team'))

sched_opp = {}
for w, games in scores.items():
    for g in games:
        m = g.get('metadata') or {}
        h, a = m.get('home_team'), m.get('away_team')
        if h and a: sched_opp.setdefault(w, {}).update({h: a, a: h})

def pts(w, pid):
    s = stats[w].get(pid)
    return score(s) if s and s.get('gms_active') else None
def proj_pts(w, pid):
    r = proj.get(w, {}).get(pid)
    return score(r['stats']) if r else None
def opp_of(w, pid):
    r = proj.get(w, {}).get(pid)
    if r and r.get('opponent'): return r['opponent']
    return sched_opp.get(w, {}).get(info.get(pid, {}).get('team'))

stat_weeks = [w for w in WEEKS if final[w]]
allowed = []
for w in stat_weeks:
    for pid, s in stats[w].items():
        pos = info.get(pid, {}).get('pos'); o = opp_of(w, pid)
        if pos in POS and s.get('gms_active') and o: allowed.append((w, o, pos, pts(w, pid)))
allowed = pd.DataFrame(allowed, columns=['week', 'def', 'pos', 'pts'])
by_def_week = allowed.groupby(['week', 'def', 'pos']).pts.sum().reset_index()

def dvp(def_team, pos, upto):
    d = by_def_week[(by_def_week.week < upto) & (by_def_week.pos == pos)]
    if d.empty or def_team is None: return np.nan
    lg = d.groupby('def').pts.mean().mean(); mine = d[d['def'] == def_team].pts.mean()
    return mine / lg if lg and not np.isnan(mine) else np.nan

def history(pid, upto):
    g = [w for w in stat_weeks if w < upto and pts(w, pid) is not None]
    if not g: return dict(gp=0, avg=np.nan, last=np.nan, opp=np.nan, snap=np.nan, ratio=np.nan, sd=np.nan)
    p = [pts(w, pid) for w in g]; opps, snaps, ratios = [], [], []
    for w in g:
        s = stats[w][pid]
        opps.append(s.get('rec_tgt', 0) + s.get('rush_att', 0) + 0.25 * s.get('pass_att', 0))
        if s.get('tm_off_snp'): snaps.append(s.get('off_snp', 0) / s['tm_off_snp'])
        pr = proj_pts(w, pid)
        if pr and pr > 1: ratios.append(pts(w, pid) / pr)
    return dict(gp=len(g), avg=np.mean(p), last=p[-1], opp=np.mean(opps), snap=np.mean(snaps) if snaps else np.nan,
                ratio=np.mean(ratios) if ratios else np.nan, sd=np.std(p) if len(p) > 1 else np.nan)

FIRST_PROJ_WEEK = 2                                     # as in the fixture
rows = []
for w in stat_weeks:
    if w < FIRST_PROJ_WEEK: continue
    for pid in proj[w]:
        pos = info[pid]['pos']; y = pts(w, pid)
        if pos not in POS or y is None: continue
        rows.append(dict(proj=proj_pts(w, pid), pos=POSI[pos], posname=pos, dvp=dvp(opp_of(w, pid), pos, w), **history(pid, w), y=y, week=w, pid=pid))
df = pd.DataFrame(rows)
FEATS = ['proj', 'pos', 'dvp', 'gp', 'avg', 'last', 'opp', 'snap', 'ratio', 'sd']
base = df[df.proj.notna() & (df.proj > 0.5)]
print(df.shape, '| rows per week (projected > 0.5):', base.groupby('week').size().to_dict())

(2904, 14) | rows per week (projected > 0.5): {2: 371, 3: 375, 4: 354, 5: 336}


## Candidates and rolling-origin evaluation

Test weeks: every final week with projections and at least 2 earlier weeks of training data (the fixture notebook's rule). Week 6 joins
automatically once Monday night's game is final.

In [4]:
def gbm(**k):
    kw = dict(max_iter=200, learning_rate=0.03, max_leaf_nodes=8, min_samples_leaf=60, l2_regularization=3, random_state=0); kw.update(k)
    return HistGradientBoostingRegressor(**kw)

def candidates(tr, X):
    out = {'projection': X.proj.values}
    out['residual_gbm'] = X.proj.values + 0.3 * gbm().fit(tr[FEATS], tr.y - tr.proj).predict(X[FEATS])
    w = X.gp / (X.gp + 10)
    out['shrink_to_avg'] = ((1 - w) * X.proj + w * X.avg.fillna(X.proj)).values
    out['gbm'] = gbm().fit(tr[FEATS], tr.y).predict(X[FEATS])
    return out
NAMES = ['projection', 'residual_gbm', 'shrink_to_avg', 'gbm']

usable = sorted(int(w) for w in base.week.unique())
test_weeks = [w for w in usable if sum(1 for v in usable if v < w) >= 2]
print('final weeks', stat_weeks, '| usable', usable, '| test weeks', test_weeks, '| week 6 final:', final.get(6))
preds = []
for tw in test_weeks:
    tr, te = base[base.week < tw], base[base.week == tw]
    out = candidates(tr, te)
    p = te[['week', 'pid', 'posname', 'y', 'proj']].copy()
    for k in NAMES: p[k] = out[k]
    preds.append(p)
preds = pd.concat(preds, ignore_index=True)
for k in NAMES: preds['ae_' + k] = (preds.y - preds[k]).abs()

per_week = preds.groupby('week')[['ae_' + k for k in NAMES]].mean().rename(columns=lambda c: c[3:])
per_week['n'] = preds.groupby('week').size()
per_week['best'] = per_week[NAMES].idxmin(axis=1)
summary = pd.DataFrame({
    'mean across test weeks': per_week[NAMES].mean(),
    'mean, weeks 4-5': per_week.loc[[w for w in per_week.index if w in (4, 5)], NAMES].mean(),
    'pooled rows': preds[['ae_' + k for k in NAMES]].mean().rename(lambda c: c[3:]),
})
original = pd.Series({'projection': 4.39, 'residual_gbm': 4.43, 'shrink_to_avg': 4.47, 'gbm': 4.68}, name='fixture, weeks 4-5')
display(per_week.round(3))
display(summary.join(original).T.round(3))

final weeks [1, 2, 3, 4, 5] | usable [2, 3, 4, 5] | test weeks [4, 5] | week 6 final: False


,projection,residual_gbm,shrink_to_avg,gbm,n,best
week,,,,,,
4,4.32,4.356,4.338,4.561,354,projection
5,4.46,4.492,4.599,4.773,336,projection


,projection,residual_gbm,shrink_to_avg,gbm
mean across test weeks,4.390,4.424,4.468,4.667
"mean, weeks 4-5",4.390,4.424,4.468,4.667
pooled rows,4.388,4.422,4.465,4.664
"fixture, weeks 4-5",4.390,4.430,4.470,4.680


## Are the differences real?

Paired bootstrap over held-out player-weeks, as in the fixture notebook (positive = candidate worse than the projection).

In [5]:
rng = np.random.default_rng(0)
n = len(preds); idx = rng.integers(0, n, size=(2000, n))
rows = []
for k in NAMES[1:]:
    d = (preds['ae_' + k] - preds['ae_projection']).values
    boots = d[idx].mean(1)
    rows.append(dict(candidate=k, mean_diff_vs_projection=d.mean(), ci_low=np.quantile(boots, 0.025), ci_high=np.quantile(boots, 0.975),
                     share_of_rows_better=(d < 0).mean()))
pd.DataFrame(rows).set_index('candidate').round(3)

,mean_diff_vs_projection,ci_low,ci_high,share_of_rows_better
candidate,,,,
residual_gbm,0.034,-0.000,0.069,0.487
shrink_to_avg,0.077,0.023,0.132,0.472
gbm,0.276,0.167,0.387,0.417


## Floor / ceiling coverage

Same check as the fixture notebook: 20th / 80th percentile of actual ÷ projected per position, fitted on earlier weeks only; target 60%.

In [6]:
def ratio_q(d):
    d = d[d.proj > 3]
    return {p: (float(np.quantile(g.y / g.proj, 0.2)), float(np.quantile(g.y / g.proj, 0.8))) for p, g in d.groupby('posname') if len(g) >= 25}

cov_rows = []
for tw in test_weeks:
    qq = ratio_q(base[base.week < tw]); te = base[(base.week == tw) & (base.proj > 3) & base.posname.isin(qq)]
    lo = te.proj * te.posname.map(lambda p: qq[p][0]); hi = te.proj * te.posname.map(lambda p: qq[p][1])
    inside = (te.y >= lo) & (te.y <= hi)
    cov_rows.append(dict(week=tw, n=len(te), coverage=inside.mean(), below_floor=(te.y < lo).mean(), above_ceiling=(te.y > hi).mean()))
cov = pd.DataFrame(cov_rows).set_index('week')
display(cov.round(3))
print(f'pooled coverage over test weeks: {(cov.coverage * cov.n).sum() / cov.n.sum():.3f} (target 0.60)')

,n,coverage,below_floor,above_ceiling
week,,,,
4,269,0.595,0.167,0.238
5,260,0.577,0.227,0.196


pooled coverage over test weeks: 0.586 (target 0.60)


## Did Sleeper change weeks 4-5 after the fixture was captured?

Same players, same weeks: the fixture's projections against today's, both scored against today's actuals.

In [7]:
def fx(url_ok):
    hits = [f for u, f in manifest['files'].items() if url_ok(u)]
    assert len(hits) == 1, hits
    with gzip.open(FIX / hits[0], 'rt') as fh:
        return json.load(fh)
fix_proj = {w: {r['player_id']: r for r in fx(lambda u, w=w: f'/projections/nfl/{SEASON}/{w}?' in u)} for w in (4, 5, 6)}
fix_sched = fx(lambda u: '/schedule/nfl/regular/' in u)

chg = []
for w in (4, 5):
    if not final[w]: continue
    for pid in proj[w]:
        if info[pid]['pos'] not in POS: continue
        y, a = pts(w, pid), proj_pts(w, pid)
        b = score(fix_proj[w][pid]['stats']) if pid in fix_proj[w] else None
        if y is None or a is None or b is None or a <= 0.5: continue
        chg.append(dict(week=w, fresh=a, fixture=b, y=y))
chg = pd.DataFrame(chg)
print(f'{len(chg)} player-weeks; projection changed for {(chg.fresh - chg.fixture).abs().gt(0.01).mean():.1%} of them')
print(f'MAE fresh {(chg.y - chg.fresh).abs().mean():.3f} | fixture {(chg.y - chg.fixture).abs().mean():.3f}')

690 player-weeks; projection changed for 0.0% of them
MAE fresh 4.388 | fixture 4.388


## How much does the rewrite flatter the MAE? (week 6)

The fixture holds week 6's projections as of the capture (Friday afternoon). Rows are limited to players whose game had **not** started at
capture (the fixture schedule says so) and is **final now** (today's scores endpoint), so the check works on a partial week too.

* *pre-kickoff* = fixture projection; *rewritten* = today's projection for the same week.
* **Late scratches**: players the pre-kickoff projection had above 0.5 points who then did not play. The rewritten projection hides them.
* **MAE on players who played**, both projections on the same rows.

The fixture was captured on Friday; Sunday's inactive lists came later, so the app's last pre-kickoff projection sits between the two
numbers. The gap is an upper bound on the flattery for the app's final projection, and about right for a Friday look at the matchup.

In [8]:
pre_started = {t for g in fix_sched if g['week'] == 6 and g.get('status') != 'pre_game' for t in (g['home'], g['away'])}
final_now = {t for g in scores.get(6, []) if g.get('status') == 'complete'
             for t in ((g.get('metadata') or {}).get('home_team'), (g.get('metadata') or {}).get('away_team')) if t}
teams = final_now - pre_started
print(f'week 6 games final now: {len(final_now) // 2} | teams whose game was unstarted at capture and is final now: {len(teams)}')

rw = []
for pid, r in fix_proj[6].items():
    pos = info.get(pid, {}).get('pos') or ((r.get('player') or {}).get('position'))
    team = (r.get('player') or {}).get('team') or r.get('team')
    if pos not in POS[:-1] or team not in teams: continue
    pre = score(r['stats']); new = proj_pts(6, pid); y = pts(6, pid)
    rw.append(dict(pid=pid, pos=pos, pre=pre, new=new, y=y))
rw = pd.DataFrame(rw)
pre_big = rw[rw.pre > 0.5]
scratch = pre_big[pre_big.y.isna()]
played = rw[rw.y.notna() & ((rw.pre > 0.5) | (rw.new.fillna(0) > 0.5))].fillna({'new': 0})
print(f'projected > 0.5 before kickoff: {len(pre_big)} | did not play: {len(scratch)} ({len(scratch) / max(len(pre_big), 1):.1%}), '
      f'pre-kickoff points lost to them: {scratch.pre.sum():.0f}')
print(f'rewritten projection > 0.5 but did not play: {int(((rw.new > 0.5) & rw.y.isna()).sum())}')
flat = pd.DataFrame({'pre-kickoff (fixture)': [(played.y - played.pre).abs().mean()], 'rewritten (today)': [(played.y - played.new).abs().mean()],
                     'n played': [len(played)]}, index=['MAE, week 6 rows final so far'])
flat['flattery'] = flat.iloc[:, 0] - flat.iloc[:, 1]
flat.round(3)

week 6 games final now: 10 | teams whose game was unstarted at capture and is final now: 18
projected > 0.5 before kickoff: 200 | did not play: 6 (3.0%), pre-kickoff points lost to them: 30
rewritten projection > 0.5 but did not play: 0


,pre-kickoff (fixture),rewritten (today),n played,flattery
"MAE, week 6 rows final so far",4.796,4.723,208,0.073


## Verdict, printed from the numbers

So the summary cannot drift from the tables when the notebook is re-run later in the season.

In [9]:
rank = summary['mean across test weeks'].sort_values()
winner = rank.index[0]
print('run on', FETCHED.strftime('%Y-%m-%d %H:%M UTC'), '| test weeks', test_weeks, '|', len(preds), 'player-weeks')
print('ranking (MAE):', {k: round(v, 3) for k, v in rank.items()})
print('plain projection wins' if winner == 'projection' else f'plain projection does NOT win: {winner} is lower by {rank["projection"] - rank[winner]:.3f} points')
print(f'rewrite flattery on week 6 so far: {flat.flattery.iloc[0]:.3f} points of MAE over {int(flat["n played"].iloc[0])} players; '
      f'late scratches hidden: {len(scratch)} of {len(pre_big)}')

run on 2026-10-18 23:02 UTC | test weeks [4, 5] | 690 player-weeks
ranking (MAE): {'projection': 4.39, 'residual_gbm': 4.424, 'shrink_to_avg': 4.468, 'gbm': 4.667}
plain projection wins
rewrite flattery on week 6 so far: 0.073 points of MAE over 208 players; late scratches hidden: 6 of 200


## Conclusion

**Snapshot of the run on Sunday, October 18, 2026 (week 6 not final: 10 of 14 games complete).** Re-run after Monday night's game for
the week-6 bake-off; the verdict cell above updates itself, this paragraph does not.

* **Weeks 4-5 reproduce from fresh data.** Today's projections for weeks 4 and 5 are identical to the fixture's (0 of 690 changed), and
  the bake-off gives the same 4.39 for the plain projection. The ML candidates move by up to 0.016 because the player table is rebuilt
  from fresh player records, and the ranking is unchanged: projection, residual GBM, shrink-to-average, GBM.
* **Week 6 is not a test week yet**, so the 2026 evidence is still those two weeks. The full 2025 season in
  `projection_bakeoff_2025.ipynb` is the larger sample.
* **The rewrite flatters the MAE, and the size is now measured.** For the 9 week-6 games that had not started when the fixture was captured
  and are final now, 6 of the 200 players projected above 0.5 points before kickoff (3.0%) did not play; the rewritten projection hides
  every one of them. On the 208 players who played, the Friday projection's MAE is **4.80** against **4.72** for the rewritten one:
  about **0.07 points (1.5%)** of flattery. That is larger than any gap between candidates in the 2025 season (at most 0.03), so the bake-off
  rankings should be read as "on equal, slightly hindsight-informed projections" and not as real-time accuracy. Part of the 0.07 is
  ordinary Friday-to-Sunday news the app would also have picked up by kickoff, so the flattery of the app's final projection is somewhat
  smaller.
